# 06 · Inference & Explainable Decisions
Load the deployed artifact and score **raw** applicants, with ECOA-style **adverse-action reason codes** (why an application was declined) — required for regulatory compliance and customer transparency.

In [1]:
# Make `src` importable when running from the notebooks/ folder
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np
pd.set_option('display.max_columns', 60)

In [2]:
from src.inference import LoanScorer
from src.data_loader import load_raw
scorer = LoanScorer()
applicants = load_raw().drop(columns=['Loan Status']).sample(8, random_state=7)
scored = scorer.predict(applicants)
scored[['loan_amount','number_of_defaults','salary','default_probability','decision','risk_band']]

,loan_amount,number_of_defaults,salary,default_probability,decision,risk_band
33909,24000.0,2,3048.266611,0.7236,APPROVE,High
92307,34000.0,1,2987.210292,0.4613,APPROVE,High
89107,2000.0,0,1902.619213,0.1128,APPROVE,Low
9538,56000.0,1,4677.075381,0.1815,APPROVE,Medium
31008,35000.0,0,3293.963497,0.1272,APPROVE,Low
71305,44000.0,2,2141.491970,0.3272,APPROVE,Medium
71012,47000.0,0,3160.090475,0.3130,APPROVE,Medium
62938,32000.0,0,2260.063282,0.1132,APPROVE,Low


## Per-applicant explanation

In [3]:
# Explain the highest-risk applicant in plain English
idx = scored['default_probability'].idxmax()
print(scorer.explain_text(applicants.loc[[idx]]))

Decision: APPROVE  (estimated default probability 72.4%, approve below 78.9%)


In [4]:
# Structured reasons (for logging / API responses)
import json
print(json.dumps(scorer.explain(applicants.loc[[idx]]), indent=2, default=str))

{
  "default_probability": 0.7236,
  "top_reasons": [
    {
      "feature": "location",
      "description": "Location",
      "applicant_value": "Chipinge",
      "typical_approved_value": "Harare",
      "risk_contribution": 0.4068
    },
    {
      "feature": "number_of_defaults",
      "description": "Number of previous loan defaults",
      "applicant_value": 2,
      "typical_approved_value": 0.0,
      "risk_contribution": 0.2607
    },
    {
      "feature": "job",
      "description": "Occupation",
      "applicant_value": "Nurse",
      "typical_approved_value": "Engineer",
      "risk_contribution": 0.0022
    }
  ],
  "decision": "APPROVE",
  "threshold": 0.7887716719866072
}
